In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.__version__

'2.14.0+cu132'

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, bias=True):
        super().__init__()
        self.out_channels = out_channels

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, stride, padding, bias=bias)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        return x

class DownSample(nn.Module):
    def __init__(self, kernel_size, stride, padding):
        super().__init__()
        self.maxpool = nn.MaxPool2d(kernel_size, stride, padding)

    def forward(self, x):
        x = self.maxpool(x)

        return x

class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, bias=True):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)

    def forward(self, x):
        x = self.conv(x)

        return x

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv_1 = DoubleConv(in_channels=in_channels, out_channels=64, kernel_size=3, stride=1, padding=0)
        self.downsample_1 = DownSample(kernel_size=2, stride=2, padding=0)

        self.double_conv_2 = DoubleConv(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=0)
        self.downsample_2 = DownSample(kernel_size=2, stride=2, padding=0)
        
        self.double_conv_3 = DoubleConv(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=0)
        self.downsample_3 = DownSample(kernel_size=2, stride=2, padding=0)
        
        self.double_conv_4 = DoubleConv(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=0)
        self.downsample_4 = DownSample(kernel_size=2, stride=2, padding=0)
        
        self.double_conv_5 = DoubleConv(in_channels=512, out_channels=1024, kernel_size=3, stride=1, padding=0)

        self.upsample_1 = UpSample(in_channels=1024, out_channels=512, kernel_size=2, stride=2, padding=0)
        self.double_conv_6 = DoubleConv(in_channels=1024, out_channels=512, kernel_size=3, stride=1, padding=0)
        
        self.upsample_2 = UpSample(in_channels=512, out_channels=256, kernel_size=2, stride=2, padding=0)
        self.double_conv_7 = DoubleConv(in_channels=512, out_channels=256, kernel_size=3, stride=1, padding=0)
        
        self.upsample_3 = UpSample(in_channels=256, out_channels=128, kernel_size=2, stride=2, padding=0)
        self.double_conv_8 = DoubleConv(in_channels=256, out_channels=128, kernel_size=3, stride=1, padding=0)
        
        self.upsample_4 = UpSample(in_channels=128, out_channels=64, kernel_size=2, stride=2, padding=0)
        self.double_conv_9 = DoubleConv(in_channels=128, out_channels=64, kernel_size=3, stride=1, padding=0)
        
        self.conv = nn.Conv2d(in_channels=64, out_channels=out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x = self.double_conv_1(x)
        dc_1_feature_cropped = x.clone().detach()[:, :, 88:480, 88:480]
        print("dc_1_feature_cropped.shape", dc_1_feature_cropped.shape)
        x = self.downsample_1(x)

        x = self.double_conv_2(x)
        dc_2_feature_cropped = x.clone().detach()[:, :, 40:240, 40:240]
        x = self.downsample_2(x)
        
        x = self.double_conv_3(x)
        dc_3_feature_cropped = x.clone().detach()[:, :, 16:120, 16:120]
        x = self.downsample_3(x)
        x = self.double_conv_4(x)
        dc_4_feature_cropped = x.clone().detach()[:, :, 4:60, 4:60]
        x = self.downsample_4(x)

        x = self.double_conv_5(x)

        x = self.upsample_1(x)
        x = torch.cat((dc_4_feature_cropped, x), dim=-3)
        x = self.double_conv_6(x)

        x = self.upsample_2(x)
        x = torch.cat((dc_3_feature_cropped, x), dim=-3)
        x = self.double_conv_7(x)

        x = self.upsample_3(x)
        x = torch.cat((dc_2_feature_cropped, x), dim=-3)
        x = self.double_conv_8(x)

        x = self.upsample_4(x)
        print("upsample_4 x.shape", x.shape)
        x = torch.cat((dc_1_feature_cropped, x), dim=-3)
        x = self.double_conv_9(x)

        x = self.conv(x)

        return x

In [27]:
DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else "cpu"

In [28]:
unet = UNet(in_channels=1, out_channels=10)
unet = unet.to(DEVICE)
print(unet)

UNet(
  (double_conv_1): DoubleConv(
    (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1))
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  )
  (downsample_1): DownSample(
    (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (double_conv_2): DoubleConv(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
  )
  (downsample_2): DownSample(
    (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (double_conv_3): DoubleConv(
    (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
    (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
  )
  (downsample_3): DownSample(
    (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (double_conv_4): DoubleConv(
    (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1))
    (conv2): Conv2d(512, 512, kernel_

In [29]:
x = torch.rand((1, 1, 572, 572))
x = x.to(DEVICE)

with torch.no_grad():
    output = unet(x)

print(output.shape)

dc_1_feature_cropped.shape torch.Size([1, 64, 392, 392])
upsample_4 x.shape torch.Size([1, 64, 392, 392])
torch.Size([1, 10, 388, 388])
